# BO Forge minimisation qLogEI campaign

This notebook demonstrates the v0.2 `CampaignSession` workflow for a minimisation problem.

It fills the Sobol initial design one experiment at a time, then requests one qLogEI batch and records all batch results before confirming that best-so-far means the lowest user-facing objective value.

## 1. Setup

The example config uses a batch campaign. The session object owns the config, log path, and current campaign DataFrame.

In [ ]:
from pathlib import Path
import os
import shutil
import sys

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
mpl_cache = PROJECT_ROOT / ".matplotlib-cache"
mpl_cache.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(mpl_cache))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from bo_forge.session import CampaignSession

In [ ]:
config_path = PROJECT_ROOT / "configs" / "02_simple_2d_minimise_qlogei.yaml"
seed_log_path = PROJECT_ROOT / "examples" / "02_simple_2d_minimise_qlogei_campaign_log.csv"
working_log_path = PROJECT_ROOT / "examples" / "02_simple_2d_minimise_qlogei_working_log.csv"
latest_suggestion_path = PROJECT_ROOT / "examples" / "02_simple_2d_minimise_qlogei_latest_suggestions.csv"
report_path = PROJECT_ROOT / "reports" / "02_simple_2d_minimise_qlogei_campaign_report.txt"

shutil.copyfile(seed_log_path, working_log_path)

campaign = CampaignSession.from_files(config_path=config_path, log_path=working_log_path)

In [ ]:
def simulated_defect_rate(row) -> float:
    loading = float(row["catalyst_loading"])
    temperature = float(row["cure_temperature"])
    defect_rate = 0.08 + 2.2 * (loading - 0.38) ** 2
    defect_rate += ((temperature - 128.0) / 38.0) ** 2
    defect_rate += 0.015 * np.sin(9.0 * loading)
    defect_rate += 0.01 * np.cos(temperature / 18.0)
    return float(defect_rate)


def current_log():
    campaign.reload()
    campaign.validate()
    return campaign.df


def request_one_candidate():
    suggestion = campaign.suggest_next(batch_size=1)
    suggestion.to_csv(latest_suggestion_path, index=False)
    campaign.append_suggestions(suggestion)
    return suggestion


def request_batch(batch_size: int):
    suggestions = campaign.suggest_next(batch_size=batch_size)
    suggestions.to_csv(latest_suggestion_path, index=False)
    campaign.append_suggestions(suggestions)
    return suggestions


def enter_simulated_result(suggestion):
    row = suggestion.iloc[0]
    result = simulated_defect_rate(row)
    campaign.mark_observed(str(row["row_id"]), result)
    return result


def enter_simulated_batch_results(suggestions):
    records = []
    for _, row in suggestions.iterrows():
        result = simulated_defect_rate(row)
        campaign.mark_observed(str(row["row_id"]), result)
        records.append(compact_record(row, result))
    return records


def compact_record(row, result):
    return {
        "source": row["source"],
        "iteration": int(row["iteration"]),
        "catalyst_loading": float(row["catalyst_loading"]),
        "cure_temperature": float(row["cure_temperature"]),
        "defect_rate": result,
    }

## 2. Load and summarise the current campaign

The campaign starts from two manually observed experiments. The remaining initial-design points will be suggested one at a time using Sobol. `campaign.next_action()` shows the recommended next notebook step.

In [ ]:
campaign.validate()
print("campaign.summary():")
display(campaign.summary())
print("--"*40)
print("campaign.next_action():")
display(campaign.next_action())
print("--"*40)
campaign.df

## 3. Fill the Sobol initial design

Each loop below represents one lab iteration: suggest one candidate, run one experiment, enter one result, reload the log.

In [ ]:
sobol_records = []
while len(campaign.observed_data()) < campaign.config.bo.initial_design_size:
    suggestion = request_one_candidate()
    result = enter_simulated_result(suggestion)
    df = current_log()
    sobol_records.append(compact_record(suggestion.iloc[0], result))

sobol_df = pd.DataFrame(sobol_records)
assert set(sobol_df["source"]) == {"sobol"}
sobol_df

## 4. Run one qLogEI batch BO round

Once the initial design is complete, the next session suggestion fits the GP and uses qLogEI because this notebook requests a batch.

In [ ]:
batch_suggestions = request_batch(campaign.config.bo.batch_size)
bo_records = enter_simulated_batch_results(batch_suggestions)
df = current_log()
bo_df = pd.DataFrame(bo_records)

assert campaign.config.bo.batch_size > 1
assert set(bo_df["source"]) == {"qlog_ei"}

display(batch_suggestions)
bo_df

In [ ]:
batch_suggestions = request_batch(campaign.config.bo.batch_size)
bo_records = enter_simulated_batch_results(batch_suggestions)
df = current_log()
bo_df = pd.DataFrame(bo_records)

assert campaign.config.bo.batch_size > 1
assert set(bo_df["source"]) == {"qlog_ei"}

print("batch suggestion:")
display(batch_suggestions)
print("--"*40)

print("bo_df:")
bo_df

In [ ]:
batch_suggestions = request_batch(campaign.config.bo.batch_size)
bo_records = enter_simulated_batch_results(batch_suggestions)
df = current_log()
bo_df = pd.DataFrame(bo_records)

assert campaign.config.bo.batch_size > 1
assert set(bo_df["source"]) == {"qlog_ei"}

print("batch suggestion:")
display(batch_suggestions)
print("--"*40)

print("bo_df:")
bo_df

## 5. Confirm the minimisation best

For a minimisation campaign, the best observation is the lowest user-facing objective value.

In [ ]:
df = current_log()
observed = campaign.observed_data()
values = pd.to_numeric(observed[campaign.config.objective.name])
best_index = values.idxmin()
best_row = observed.loc[
    [best_index],
    ["row_id", *campaign.config.variable_names, campaign.config.objective.name],
]

print(f"Objective direction: {campaign.config.objective.direction}")
print(f"Best means lowest {campaign.config.objective.name}: {values.min():.6f}")
display(best_row)

assert campaign.config.objective.direction == "minimize"
assert values.min() == values.cummin().iloc[-1]

## 6. Diagnostics

In [ ]:
campaign.plot_progress();

In [ ]:
campaign.plot_diagnostics();

## 7. Reports

In [ ]:
campaign.report()

In [ ]:
campaign.export_report(report_path)